In [1]:
# data and analysis lib
import polars as pl                     # fast dataframes for financial data
import numpy as np                      # Numerical computing lib
from datetime import datetime, timedelta # Date and time operations
import random

# ML libraries
import torch                            # PyTorch framework
import torch.nn as nn                   # Neural network modules
import torch.optim as optim             # Optimization algorithms
import research                         # Model building and training utilities from memlabs-research

# Visualization
import altair as alt                    # Interactive visualization lib

# data source
from ccxt import binance                          # Binance market data utilities


In [2]:
research.set_seed(42)

In [3]:
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_cols(-1)

polars.config.Config

In [4]:
# parameters for the reseach libarary

#trading pair symbol
sym = 'BTC/USDT' 

# historical data window in days (e.g. 6 months)
hist_data_window = 24 * 7 * 4 * 6
#time horizon of time series (time interval), tic rate
time_interval = '1h'
# max number of auto-regressive lags (t-1, 2, 3 etc)
max_lags = 4 #future log returns based on previous log returns, with mem up to 4 lags/tics
#forecast horizon in steps, based on the time interval
forecast_horizon = 1 
# sharpe annulized rate (so its independent of time frequency)
annualized_rate = research.sharpe_annualization_factor(time_interval, 365, 24)

In [5]:
# initialize Binance exchange
exchange = binance()

#fetch historical data as timeseries (ts)
bars = exchange.fetch_ohlcv(symbol = sym, timeframe= time_interval, limit = hist_data_window)
# 3. Skapa en Polars DataFrame
df = pl.DataFrame(
    bars, 
    schema=['timestamp', 'open', 'high', 'low', 'close', 'volume'],
    orient="row"
)

# 4. Konvertera timestamp till datetime
df = df.with_columns(
    pl.from_epoch("timestamp", time_unit="ms").alias("timestamp")
)
print(df.head())

shape: (5, 6)
┌─────────────────────┬──────────┬──────────┬──────────┬──────────┬────────────┐
│ timestamp           ┆ open     ┆ high     ┆ low      ┆ close    ┆ volume     │
│ ---                 ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---        │
│ datetime[μs]        ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64        │
╞═════════════════════╪══════════╪══════════╪══════════╪══════════╪════════════╡
│ 2026-04-27 05:00:00 ┆ 78924.21 ┆ 78938.0  ┆ 77562.63 ┆ 77785.54 ┆ 1805.78745 │
│ 2026-04-27 06:00:00 ┆ 77785.55 ┆ 77871.9  ┆ 77465.05 ┆ 77700.12 ┆ 641.73182  │
│ 2026-04-27 07:00:00 ┆ 77700.12 ┆ 77734.79 ┆ 77562.08 ┆ 77606.6  ┆ 743.19766  │
│ 2026-04-27 08:00:00 ┆ 77606.59 ┆ 77816.18 ┆ 77553.23 ┆ 77804.34 ┆ 465.54163  │
│ 2026-04-27 09:00:00 ┆ 77804.34 ┆ 77980.0  ┆ 77700.0  ┆ 77900.01 ┆ 578.50773  │
└─────────────────────┴──────────┴──────────┴──────────┴──────────┴────────────┘


In [6]:
# 2. Skapa interaktiv graf med Altair
chart = alt.Chart(df).mark_line().encode(
    x=alt.X('timestamp:T', title='datetime', axis = alt.Axis(format='%b %d', tickCount='day')),
    y=alt.Y('close:Q', title='close', scale=alt.Scale(zero=False)),
    tooltip=['timestamp', 'close']
).properties(
    title='BTC/USDT 1h close',
    width=800,
    height=400
).interactive() # .interactive() ger dig zoom och pan

chart.show()

alt.Chart(...)

### Feature Engineering

In [7]:
price_time_series = pl.DataFrame({'price':[100.0,120.0,100.0]})
research.plot_column(price_time_series, 'price')

alt.Chart(...)

In [8]:
price_time_series.with_columns(
    pl.col('price').diff().alias('delta'),
    ((pl.col('price')-pl.col('price').shift())/pl.col('price').shift()).alias('return'),
    (pl.col('price')/pl.col('price').shift()).log().alias('log_retun'),
)


price,delta,return,log_retun
f64,f64,f64,f64
100.0,null,null,null
120.0,20.0,0.2,0.182322
100.0,-20.0,-0.166667,-0.182322


### Create target and lagged target

In [9]:
df = df.with_columns((pl.col('close')/pl.col('close').shift(forecast_horizon)).log().alias('close_log_return'))
df

timestamp,open,high,low,close,volume,close_log_return
datetime[μs],f64,f64,f64,f64,f64,f64
2026-04-27 05:00:00,78924.21,78938.0,77562.63,77785.54,1805.78745,null
2026-04-27 06:00:00,77785.55,77871.9,77465.05,77700.12,641.73182,-0.001099
2026-04-27 07:00:00,77700.12,77734.79,77562.08,77606.6,743.19766,-0.001204
2026-04-27 08:00:00,77606.59,77816.18,77553.23,77804.34,465.54163,0.002545
2026-04-27 09:00:00,77804.34,77980.0,77700.0,77900.01,578.50773,0.001229
…,…,…,…,…,…,…
2026-06-07 16:00:00,62093.99,62332.0,61966.0,62083.64,482.9603,-0.000167
2026-06-07 17:00:00,62083.64,62330.0,62046.62,62160.01,573.97699,0.001229
2026-06-07 18:00:00,62160.01,62296.0,62022.0,62030.14,373.61361,-0.002091


In [10]:
target = 'close_log_return'
lr = pl.col(target)
df = df.with_columns(
    lr.shift(forecast_horizon * 1).alias(f'{target}_lag_1'),
    lr.shift(forecast_horizon * 2).alias(f'{target}_lag_2'),
    lr.shift(forecast_horizon * 3).alias(f'{target}_lag_3'),
    lr.shift(forecast_horizon * 4).alias(f'{target}_lag_4'),
)
#df = research.add_lags(df, target, max_lags, forecast_horizon)
df

timestamp,open,high,low,close,volume,close_log_return,close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-04-27 05:00:00,78924.21,78938.0,77562.63,77785.54,1805.78745,null,null,null,null,null
2026-04-27 06:00:00,77785.55,77871.9,77465.05,77700.12,641.73182,-0.001099,null,null,null,null
2026-04-27 07:00:00,77700.12,77734.79,77562.08,77606.6,743.19766,-0.001204,-0.001099,null,null,null
2026-04-27 08:00:00,77606.59,77816.18,77553.23,77804.34,465.54163,0.002545,-0.001204,-0.001099,null,null
2026-04-27 09:00:00,77804.34,77980.0,77700.0,77900.01,578.50773,0.001229,0.002545,-0.001204,-0.001099,null
…,…,…,…,…,…,…,…,…,…,…
2026-06-07 16:00:00,62093.99,62332.0,61966.0,62083.64,482.9603,-0.000167,0.00304,0.003323,-0.003245,-0.011585
2026-06-07 17:00:00,62083.64,62330.0,62046.62,62160.01,573.97699,0.001229,-0.000167,0.00304,0.003323,-0.003245
2026-06-07 18:00:00,62160.01,62296.0,62022.0,62030.14,373.61361,-0.002091,0.001229,-0.000167,0.00304,0.003323


In [11]:
df = df.drop_nulls()

In [71]:
research.plot_distribution(df, target, no_bins=100)

alt.Chart(...)

In [ ]:
research.plot_distribution(df, 'close', no_bins=100) # why not to look at prices, since models makes no assuptions this distribution will be hard to model

alt.Chart(...)

### Build Model

In [12]:
class LinearModel(nn.Module):
    def __init__(self, input_features):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(input_features, 1) # 1 output
    
    def forward(self, x):
        return self.linear(x)

### Complexity of model

In [13]:
input_features = 1

linear_model = LinearModel(input_features)

research.print_model_info(linear_model, "Linear Model")
research.total_model_params(linear_model)


Linear Model

Architecture:
  LinearModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)

Parameter Count:
  Total parameters:      2
  Trainable parameters:  2



2

### Split data by time

In [14]:
# this is to avoild data leakage

features = ['close_log_return_lag_1']
target = 'close_log_return'
test_size = 0.25 #model will train 75% and predict 25%

In [15]:
len(df)

995

In [16]:
int(len(df) * test_size)

248

In [17]:
split_idx = int(len(df) * (1-test_size))
split_idx

746

In [18]:
df_train, df_test = df[:split_idx], df[split_idx:]

In [98]:
df_train

timestamp,open,high,low,close,volume,close_log_return,close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-04-27 22:00:00,76786.81,77082.78,76763.71,77066.42,364.86359,0.003635,-0.002259,0.001071,0.000573,0.001183
2026-04-27 23:00:00,77066.42,77450.0,77018.56,77371.32,438.27302,0.003949,0.003635,-0.002259,0.001071,0.000573
2026-04-28 00:00:00,77371.32,77478.0,77210.92,77379.62,327.26244,0.000107,0.003949,0.003635,-0.002259,0.001071
2026-04-28 01:00:00,77379.62,77394.1,77038.1,77084.68,404.77602,-0.003819,0.000107,0.003949,0.003635,-0.002259
2026-04-28 02:00:00,77084.69,77227.03,76672.56,76890.93,437.7548,-0.002517,-0.003819,0.000107,0.003949,0.003635
…,…,…,…,…,…,…,…,…,…,…
2026-05-28 09:00:00,73325.88,73560.0,73261.6,73541.24,547.18862,0.002933,-0.002345,0.004026,0.002898,-0.002614
2026-05-28 10:00:00,73541.25,73616.0,73349.91,73400.01,495.53335,-0.001922,0.002933,-0.002345,0.004026,0.002898
2026-05-28 11:00:00,73400.01,73532.01,73319.36,73505.66,383.87102,0.001438,-0.001922,0.002933,-0.002345,0.004026


In [99]:
df_test

timestamp,open,high,low,close,volume,close_log_return,close_log_return_lag_1,close_log_return_lag_2,close_log_return_lag_3,close_log_return_lag_4
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-05-28 14:00:00,73022.01,73573.21,72582.82,73033.06,2189.6201,0.000151,-0.006769,0.000168,0.001438,-0.001922
2026-05-28 15:00:00,73033.06,73180.01,72700.0,72928.73,710.64433,-0.00143,0.000151,-0.006769,0.000168,0.001438
2026-05-28 16:00:00,72928.73,73440.67,72741.4,73440.67,821.59668,0.006995,-0.00143,0.000151,-0.006769,0.000168
2026-05-28 17:00:00,73440.66,73734.72,73316.31,73554.82,737.45631,0.001553,0.006995,-0.00143,0.000151,-0.006769
2026-05-28 18:00:00,73554.81,73749.69,73514.57,73652.88,517.48323,0.001332,0.001553,0.006995,-0.00143,0.000151
…,…,…,…,…,…,…,…,…,…,…
2026-06-07 15:00:00,61905.49,62219.92,61674.51,62093.99,527.33835,0.00304,0.003323,-0.003245,-0.011585,0.004023
2026-06-07 16:00:00,62093.99,62332.0,61966.0,62083.64,482.9603,-0.000167,0.00304,0.003323,-0.003245,-0.011585
2026-06-07 17:00:00,62083.64,62330.0,62046.62,62160.01,573.97699,0.001229,-0.000167,0.00304,0.003323,-0.003245


In [19]:
x_train = torch.tensor(df_train[features].to_numpy(), dtype=torch.float32)
x_test = df_test[features].to_torch().float()
y_train = torch.tensor(df_train[target].to_numpy(), dtype=torch.float32)
y_test = df_test[target].to_torch().float()

In [ ]:
x_train # in shape n:1 Y_train is opposite

In [20]:
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1) #can be done: research.timeseries_train_test_split(df, features, target, test_size)

### Batch Gradient Descent

In [21]:
# hyper parameters
no_epochs = 1000 * 5
lr = 0.0005 # landing rate, how mutch we adjust the weight by.

# create model
model = LinearModel(len(features))

# loss function
criterion = nn.MSELoss() # mean squared error

# optimizer
optimizer = optim.Adam(model.parameters(), lr = lr)

print("\nTraining model...")

for epoch in range(no_epochs):
    # forward pass
    y_hat = model(x_train)
    loss = criterion(y_hat, y_train)

    # backward pass
    optimizer.zero_grad()  # 1. clear old gradients
    loss.backward()        # 2. compute new gradients
    optimizer.step()       # 3. update weights

    # check for imporvement
    train_loss = loss.item()

    # logging
    if (epoch + 1) % 500 == 0:
        print(f"Epoch [{epoch+1}/{no_epochs}], Loss: {train_loss:.6f}")

print("\nLearned parameters") # what are the weights and biases that we have learned

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}:\n{param.data.numpy()}")

# Evaluation
model.eval()
with torch.no_grad():
    y_hat = model(x_test)
    test_loss = criterion(y_hat, y_test)
    print(f"\nTest Loss: {test_loss.item():.6f}, Train Loss: {train_loss:.6f}")


Training model...
Epoch [500/5000], Loss: 0.469068
Epoch [1000/5000], Loss: 0.234745
Epoch [1500/5000], Loss: 0.101801
Epoch [2000/5000], Loss: 0.036174
Epoch [2500/5000], Loss: 0.009757
Epoch [3000/5000], Loss: 0.001814
Epoch [3500/5000], Loss: 0.000215
Epoch [4000/5000], Loss: 0.000024
Epoch [4500/5000], Loss: 0.000012
Epoch [5000/5000], Loss: 0.000012

Learned parameters
linear.weight:
[[0.18327431]]
linear.bias:
[-1.3367566e-05]

Test Loss: 0.000041, Train Loss: 0.000012


### Test Trading Performance

In [ ]:
trade_results = pl.DataFrame({
    'y_hat': y_hat.squeeze(),
    'y': y_test.squeeze()
}).with_columns(
    (pl.col('y_hat').sign()==pl.col('y').sign()).alias('is_won'),
    pl.col('y_hat').sign().alias('signal'),
).with_columns(
    (pl.col('signal') * pl.col('y')).alias('trade_log_return')
).with_columns(
    pl.col('trade_log_return').cum_sum().alias('equity_curve') # acumulated returns
)
trade_results


y_hat,y,is_won,signal,trade_log_return,equity_curve
f32,f32,bool,f32,f32,f32
0.00025,0.000168,true,1.0,0.000168,0.000168
0.000017,-0.006769,false,1.0,-0.006769,-0.006601
-0.001254,0.000151,false,-1.0,-0.000151,-0.006753
0.000014,-0.00143,false,1.0,-0.00143,-0.008182
-0.000275,0.006995,false,-1.0,-0.006995,-0.015177
…,…,…,…,…,…
0.000544,-0.000167,false,1.0,-0.000167,0.038276
-0.000044,0.001229,false,-1.0,-0.001229,0.037046
0.000212,-0.002091,false,1.0,-0.002091,0.034955


In [26]:
research.plot_column(trade_results, 'equity_curve')

alt.Chart(...)

In [27]:
trade_results = trade_results.with_columns(
    (pl.col('equity_curve') - pl.col('equity_curve').cum_max()).alias('drawdown_log')
)
trade_results

y_hat,y,is_won,signal,trade_log_return,equity_curve,drawdown_log
f32,f32,bool,f32,f32,f32,f32
0.00025,0.000168,true,1.0,0.000168,0.000168,0.0
0.000017,-0.006769,false,1.0,-0.006769,-0.006601,-0.006769
-0.001254,0.000151,false,-1.0,-0.000151,-0.006753,-0.006921
0.000014,-0.00143,false,1.0,-0.00143,-0.008182,-0.00835
-0.000275,0.006995,false,-1.0,-0.006995,-0.015177,-0.015345
…,…,…,…,…,…,…
0.000544,-0.000167,false,1.0,-0.000167,0.038276,-0.056949
-0.000044,0.001229,false,-1.0,-0.001229,0.037046,-0.058178
0.000212,-0.002091,false,1.0,-0.002091,0.034955,-0.06027


In [ ]:
max_drawdown_log = trade_results['drawdown_log'].min()
max_drawdown_log

-0.060393285006284714

In [29]:
drawdown_pct = np.exp(max_drawdown_log) - 1 
drawdown_pct

np.float64(-0.05860577546298962)

In [30]:
equity_peak = 1000
equity_peak * drawdown_pct

np.float64(-58.60577546298962)

In [31]:
win_rate = trade_results['is_won'].mean()
win_rate

0.4819277108433735

In [33]:
avg_win = trade_results.filter(pl.col('is_won')==True)['trade_log_return'].mean()
avg_loss = trade_results.filter(pl.col('is_won')==False)['trade_log_return'].mean()
ev = win_rate * avg_win + (1 - win_rate) * avg_loss
ev

0.00017354590614836705

In [34]:
total_log_retun = trade_results['trade_log_return'].sum()
total_log_retun

0.043212924152612686

In [35]:
compound_return = np.exp(total_log_retun)
compound_return

np.float64(1.0441601981082804)

In [36]:
# example
1000*compound_return

np.float64(1044.1601981082804)

In [37]:
equity_trough = trade_results['equity_curve'].min()
equity_trough

-0.03789125755429268

In [38]:
equity_peak = trade_results['equity_curve'].max()
equity_peak

0.09522440284490585

In [39]:
std = trade_results['trade_log_return'].std() # risk i.e standard deviation
std

0.006280259229242802

In [ ]:
# reward to risk
sharpe = ev / std * annualized_rate # annualized rate is the sqrt of number of trading periods
sharpe

np.float64(2.586359291741848)

In [42]:
research.eval_model_performance(y_test, y_hat, features, target, annualized_rate)

{'features': 'close_log_return_lag_1',
 'target': 'close_log_return',
 'no_trades': 249,
 'win_rate': 0.4819277108433735,
 'avg_win': 0.004547154393837142,
 'avg_loss': -0.003894927105655144,
 'best_trade': 0.023593146353960037,
 'worst_trade': -0.020420484244823456,
 'ev': 0.00017354590614836705,
 'std': 0.006280259229242802,
 'total_log_return': 0.043212924152612686,
 'compound_return': np.float64(1.0441601981082804),
 'max_drawdown': -0.060393285006284714,
 'equity_trough': -0.03789125755429268,
 'equity_peak': 0.09522440284490585,
 'sharpe': np.float64(2.586359291741851)}